# Atari Pong with frame preprocessing

This example uses standard [Gymnasium Atari (ALE)](https://gymnasium.farama.org/environments/atari/) envs - mouse-env does not change the games and has no Atari-specific code. Atari is configured with two general `EnvConfig` knobs: build the env in an `env_fn` factory (`gym.make` + `gymnasium.wrappers.AtariPreprocessing`), and set `observation_kind="image"` to route the frame to the image channel. Register ALE yourself with `gymnasium.register_envs(ale_py)`.

In [ ]:
import ale_py
import gymnasium as gym
import numpy as np
from gymnasium.wrappers import AtariPreprocessing

from mouse_envs import EnvConfig, make_vector_env

gym.register_envs(ale_py)

## Configuration

Requires the optional `atari` extra: `pip install 'mouse-env[atari]'` (`ale_py` + `opencv-python`, which `AtariPreprocessing` needs for frame resizing). ALE ROMs ship with `ale_py`.

`frameskip=1` in the factory disables the base env's frame skipping so `AtariPreprocessing` owns it.

In [ ]:
def make_pong():
    env = gym.make("ALE/Pong-v5", frameskip=1, max_episode_steps=10000)
    return AtariPreprocessing(env, frame_skip=4, screen_size=84, noop_max=30)


cfg = EnvConfig(
    id="ALE/Pong-v5",
    seed=0,
    num_envs=4,
    max_episode_steps=10000,
    observation_kind="image",
    env_fn=make_pong,
)
env = make_vector_env(cfg)

## First step and observation shape

After preprocessing, each env emits an 84×84 frame in its native shape. `env.obs_key` tells you which observation channel to read.

In [ ]:
result, metrics = env.step(env.sample_random_actions())

print(f"obs key:   {env.obs_key}")
batch_obs = np.stack([r["observation"]["image"].numpy() for r in result])
print(f"obs shape: {batch_obs.shape}")  # (num_envs, 84, 84)

## Show a preprocessed frame

Each observation is already an 84×84 grayscale frame — exactly what the agent sees after `AtariPreprocessing`.

In [ ]:
import matplotlib.pyplot as plt

fig, axes = plt.subplots(1, cfg.num_envs, figsize=(3 * cfg.num_envs, 3))
for ax, frame in zip(np.atleast_1d(axes), batch_obs):
    ax.imshow(frame, cmap="gray", vmin=0, vmax=255)
    ax.axis("off")
fig.tight_layout()
plt.show()

## Short follow-up rollout

In [ ]:
for _ in range(10):
    env.step(env.sample_random_actions())

env.close()